# Importando as Bibliotecas

In [6]:
# Manipulação de dados
import re
import numpy as np
import pandas as pd

# Visualização
import matplotlib.pyplot as plt

# NLP - NLTK
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import SnowballStemmer

# Recursos do NLTK
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
nltk.download("stopwords", quiet=True)
nltk.download("rslp", quiet=True)

# Preparação e validação dos dados
from sklearn.model_selection import (
    train_test_split,
    GridSearchCV,
    StratifiedKFold,
    learning_curve
)

# Representação textual
from sklearn.feature_extraction.text import TfidfVectorizer

# Modelos de Machine Learning
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC

# Pipeline
from sklearn.pipeline import Pipeline

# Avaliação
from sklearn.metrics import (
    classification_report,
    ConfusionMatrixDisplay,
    RocCurveDisplay
)

print("Bibliotecas importadas com sucesso!")

Bibliotecas importadas com sucesso!


# Leitura da Base de dados

In [2]:
dados = pd.read_csv("../data/dados_spam.csv", index_col=0)
dados.head()

,categoria,texto
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


# Avaliação do Modelo com TF-IDF

In [3]:
ABBREVIATIONS = {
    # Conversa / SMS
    "pls": "please", "plz": "please", "thx": "thanks", "ty": "thank you", "u": "you", "ur": "your", "r": "are",
    "ya": "you", "msg": "message", "txt": "text", "idk": "i do not know", "imo": "in my opinion", "imho": "in my humble opinion",
    "asap": "as soon as possible", "rn": "right now", "btw": "by the way", "fyi": "for your information", "brb": "be right back",
    "ttyl": "talk to you later", "lol": "laughing out loud", "omg": "oh my god",
    # Tempo / urgência
    "tmr": "tomorrow", "tmrw": "tomorrow", "tdy": "today", "2day": "today", "2nite": "tonight", "nite": "night", "wk": "week",
    "wks": "weeks", "hr": "hour", "hrs": "hours", "min": "minute", "mins": "minutes",
    # Financeiro
    "acct": "account", "accts": "accounts", "txn": "transaction", "txns": "transactions", "transac": "transaction", "amt": "amount",
    "bal": "balance", "avl": "available", "stmt": "statement", "stmts": "statements", "ref": "reference",
    # Pagamentos
    "pymt": "payment", "pymts": "payments", "pmt": "payment", "pmts": "payments", "paymt": "payment",
    # Promoções / spam
    "promo": "promotion", "promos": "promotions", "disc": "discount", "pct": "percent", "exp": "expires", "expd": "expired",
    "offer": "offer", "offers": "offers",
    # Contato
    "ph": "phone", "tel": "telephone", "mob": "mobile", "addr": "address", "info": "information",
}

In [4]:
STOP_WORDS = set(stopwords.words("english"))

NEGATIONS = {
    "no", "not", "nor", "never", "neither",
    "none", "nobody", "nothing", "nowhere",
    "cannot", "without",

    # Contrações negativas
    "don't", "doesn't", "didn't",
    "isn't", "aren't", "wasn't", "weren't",
    "haven't", "hasn't", "hadn't",
    "won't", "wouldn't", "can't", "couldn't",
    "shouldn't", "mustn't", "needn't",
}

STOP_WORDS_EN = STOP_WORDS - NEGATIONS

In [7]:
stemmer = SnowballStemmer("english")

In [9]:
def tratamento_tfidf(texto):
    # 1. Lowercase
    texto = texto.lower()
    # 2. Remover HTML
    texto = re.sub(r'<[^>]+>', " ", texto)
    # 3. Remover URLs
    texto = re.sub(r'https?://\S+|www\.\S+', " ", texto)
    # 4. Remover E-mails
    texto = re.sub(r'\b[\w\.-]+@[\w\.-]+\.\w+\b', " ", texto)
    # 5. Remover telefones
    texto = re.sub(r'\+?\d[\d\s\-\(\)]{7,}\d', " ", texto)
    # 6. Remover Dinheiro
    texto = re.sub(r'(r\$|\$|€|£)\s?\d+(?:[.,]\d+)?', " ", texto)
    # 7. Substituindo as abreviações
    palavras = [ABBREVIATIONS.get(p,p) for p in texto.split()]
    texto = " ".join(palavras)
    # 7. Remove caracteres indesejados
    texto = re.sub(r'[^a-záàâãéêèíìîóôõöúùûüç\s.,!?]', " ", texto)
    # 8. Normaliza espaços
    texto = re.sub(r'\s+', " ", texto)
    # 9 tokenizacao
    tokens = word_tokenize(texto, language="english")
    # 10. StopWords + Stemming
    resultado = []
    for token in tokens:
        if (token.isalpha() and token not in STOP_WORDS_EN):
            token_stemmer = stemmer.stem(token)
            resultado.append(token_stemmer)
    
    return " ".join(resultado)

In [10]:
df_tfidf = dados.copy()
df_tfidf["texto_tratado"] = df_tfidf["texto"].apply(tratamento_tfidf)
df_tfidf[["texto", "texto_tratado"]].head()

,texto,texto_tratado
0,"Go until jurong point, crazy.. Available only ...",go jurong point crazi avail bugi n great world...
1,Ok lar... Joking wif u oni...,ok lar joke wif oni
2,Free entry in 2 a wkly comp to win FA Cup fina...,free entri wkli comp win fa cup final tkts st ...
3,U dun say so early hor... U c already then say...,dun say earli hor c alreadi say
4,"Nah I don't think he goes to usf, he lives aro...",nah think goe usf live around though


## Divisão dados TF-IDF

In [11]:
X = df_tfidf["texto_tratado"]
y = df_tfidf["categoria"]

X_treino, X_teste_tfidf, y_treino, y_teste_tfidf = train_test_split(X, y, stratify=y, test_size=0.2, random_state=42)

## Carregando modelo e previsão

In [12]:
modelo_tfidf = joblib.load("../models/modelo_tfidf_final.pkl")

In [13]:
previsao_tfidf = modelo_tfidf.predict(X_teste_tfidf)

# Avaliação modelo BoW + Ngrams

In [14]:
df_bow_ngram = dados.copy()

df_bow_ngram["texto_tratado"] = df_bow_ngram["texto"].apply(tratamento_tfidf)
df_bow_ngram[["texto", "texto_tratado"]].head()

,texto,texto_tratado
0,"Go until jurong point, crazy.. Available only ...",go jurong point crazi avail bugi n great world...
1,Ok lar... Joking wif u oni...,ok lar joke wif oni
2,Free entry in 2 a wkly comp to win FA Cup fina...,free entri wkli comp win fa cup final tkts st ...
3,U dun say so early hor... U c already then say...,dun say earli hor c alreadi say
4,"Nah I don't think he goes to usf, he lives aro...",nah think goe usf live around though


## Divisão dos dados

In [15]:
X = df_bow_ngram["texto_tratado"]
y = df_bow_ngram["categoria"]

X_treino, X_teste_bow_ngram, y_treino, y_teste_bow_ngram = train_test_split(X, y, stratify=y, test_size=0.2, random_state=42)

## Carregando modelo + previsão

In [16]:
modelo_bow_ngram = joblib.load("../models/modelo_bow_ngram_final.pkl")

In [17]:
previsao_bow_ngram = modelo_bow_ngram.predict(X_teste_bow_ngram)

# Avaliação Modelo Hugging-Face